In [1]:
import pandas as pd
import numpy as np

## Unique Values, Value Counts, and Membership

These methods extract information about the **values** contained in a Series.

### `unique`

Returns an array of the unique values in a Series.

- Order reflects first appearance in the data — not sorted.
- Call `.sort()` on the result afterward if sorted order is needed.

### `value_counts`

Returns a Series where:

- **Index** = unique values.
- **Values** = frequency counts.
- Sorted by count in **descending** order by default.

Also available as a top-level function `pd.value_counts(array)` for use with NumPy arrays or other Python sequences. Pass `sort=False` to preserve the original value order.

### `isin`

Performs a **vectorized membership check** — returns a boolean Series indicating whether each element is in a given set of values.

- Useful for filtering a Series or DataFrame down to a subset of values.

### `Index.get_indexer`

Maps each value in one array to its integer position in another array of **distinct** values.

- Useful for data alignment and join-type operations.
- Returns `-1` for values not found in the target.

### Method Reference

| Method | Description |
|--------|-------------|
| `unique` | Array of unique values in a Series, in order of first appearance |
| `value_counts` | Series of unique values (index) and their frequencies (values), sorted descending |
| `isin` | Boolean array — `True` where element is in the passed sequence |
| `get_indexer` | Integer index positions mapping one array of values into another array of distinct values |

## Key Points

- `unique` returns distinct values in order of appearance, not sorted.
- `value_counts` sorts by frequency (descending) by default.
- `pd.value_counts(array)` is a top-level version usable on any array-like.
- `isin` returns a boolean mask — use it to filter rows.
- `get_indexer` maps values to their positions in a reference array of distinct values.

In [2]:
obj = pd.Series(["c", "a", "d", "a", "a", "b", "b", "c", "c"])

obj

0    c
1    a
2    d
3    a
4    a
5    b
6    b
7    c
8    c
dtype: object

In [3]:
uniques = obj.unique()

uniques

array(['c', 'a', 'd', 'b'], dtype=object)

In [4]:
obj.value_counts()

c    3
a    3
b    2
d    1
Name: count, dtype: int64

In [5]:
pd.value_counts(obj.to_numpy(), sort=False)  # top-level function, preserves value order

/tmp/ipykernel_41850/599432247.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(obj.to_numpy(), sort=False)  # top-level function, preserves value order


c    3
a    3
d    1
b    2
Name: count, dtype: int64

In [6]:
mask = obj.isin(["b", "c"])

mask

0     True
1    False
2    False
3    False
4    False
5     True
6     True
7     True
8     True
dtype: bool

In [7]:
obj[mask]

0    c
5    b
6    b
7    c
8    c
dtype: object

In [8]:
to_match = pd.Series(["c", "a", "b", "b", "c", "a"])
unique_vals = pd.Series(["c", "b", "a"])

indices = pd.Index(unique_vals).get_indexer(to_match)

indices

array([0, 2, 1, 1, 0, 2])

## Value Counts Across DataFrame Columns

### Per-Column Value Counts with `apply`

To compute value counts for **every column** in a DataFrame at once, use `apply` with `pd.value_counts`:

```python
data.apply(pd.value_counts).fillna(0)
```

- Row labels in the result are all distinct values that appear across any column.
- `fillna(0)` replaces `NaN` (value didn't appear in that column) with `0`.

### `DataFrame.value_counts`

Unlike the per-column approach, `DataFrame.value_counts()` treats **each row as a tuple** and counts how many times each distinct combination of values appears.

- Result has a **hierarchical (MultiIndex)** index representing each distinct row combination.
- Useful for finding duplicate rows or counting combinations of values.

## Key Points

- `data.apply(pd.value_counts).fillna(0)` → frequency table across all columns.
- `DataFrame.value_counts()` → counts distinct **row combinations**, not individual column values.
- The two methods answer different questions: per-column frequencies vs. whole-row frequencies.

In [9]:
data = pd.DataFrame({
    "Qu1": [1, 3, 4, 3, 4],
    "Qu2": [2, 3, 1, 2, 3],
    "Qu3": [1, 5, 2, 4, 4]
})

data

,Qu1,Qu2,Qu3
0,1,2,1
1,3,3,5
2,4,1,2
3,3,2,4
4,4,3,4


In [10]:
data["Qu1"].value_counts().sort_index()

Qu1
1    1
3    2
4    2
Name: count, dtype: int64

In [11]:
data.apply(pd.value_counts).fillna(0)  # frequency table for all columns

/tmp/ipykernel_41850/1880142476.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  data.apply(pd.value_counts).fillna(0)  # frequency table for all columns


,Qu1,Qu2,Qu3
1,1.0,1.0,1.0
2,0.0,2.0,1.0
3,2.0,2.0,0.0
4,2.0,0.0,2.0
5,0.0,0.0,1.0


In [12]:
data = pd.DataFrame({"a": [1, 1, 1, 2, 2], "b": [0, 0, 1, 0, 0]})

data

,a,b
0,1,0
1,1,0
2,1,1
3,2,0
4,2,0


In [13]:
data.value_counts()  # counts distinct row combinations

a  b
1  0    2
2  0    2
1  1    1
Name: count, dtype: int64